# Data Preprocessing
Assumes you have ran main.py and extracted pose data. This will combine the pose and EMG data, perform filtering, smoothing, etc. and put everything into a .csv file ready to be directly trained on.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import os, warnings
warnings.filterwarnings('ignore')

%matplotlib inline 
plt.rcParams['figure.figsize'] = (12, 4)

DISPLAY_INTERMEDIATE = True
recordings_dir = 'recordings'
time_stamp = '20260425_013519'

# Helper Functions

In [7]:
L1_FIXED = 14 * 0.0254    # 0.3556 m (measured upper arm)
L2_FIXED = 10 * 0.0254    # 0.254 m (measured forearm)

def resolve_sign(candidates_func, prev_val):
    """
    candidates_func returns two possible angle values (in radians).
    Pick the one closest (circularly) to prev_val.
    """
    val1, val2 = candidates_func()
    # Wrap to [-pi, pi)
    val1 = np.arctan2(np.sin(val1), np.cos(val1))
    val2 = np.arctan2(np.sin(val2), np.cos(val2))
    
    # Circular difference to prev_val
    diff1 = np.abs(np.arctan2(np.sin(val1 - prev_val), np.cos(val1 - prev_val)))
    diff2 = np.abs(np.arctan2(np.sin(val2 - prev_val), np.cos(val2 - prev_val)))
    
    return val1 if diff1 <= diff2 else val2

def compute_joint_angles_from_data(pose_df, L1=L1_FIXED):
    x1 = pose_df['elbow_x'] - pose_df['shoulder_x']
    y1 = pose_df['elbow_y'] - pose_df['shoulder_y']
    z1 = pose_df['elbow_z'] - pose_df['shoulder_z']
    
    x2 = pose_df['bracelet_x'] - pose_df['shoulder_x']
    y2 = pose_df['bracelet_y'] - pose_df['shoulder_y']
    z2 = pose_df['bracelet_z'] - pose_df['shoulder_z']
    
    n = len(pose_df)
    q1 = np.zeros(n); q2 = np.zeros(n); q3 = np.zeros(n); q4 = np.zeros(n)
    
    prev_q1, prev_q2, prev_q3, prev_q4 = 0.0, np.pi/2, 0.0, np.pi/2
    
    for i in range(n):
        base_q1 = np.arctan2(y1.iloc[i], x1.iloc[i])
        q1[i] = resolve_sign(lambda: (base_q1, np.arctan2(-y1.iloc[i], x1.iloc[i])), prev_q1)
        
        r_xy = np.sqrt(x1.iloc[i]**2 + y1.iloc[i]**2)
        q2[i] = resolve_sign(lambda: (np.arctan2(r_xy, z1.iloc[i]), np.arctan2(-r_xy, z1.iloc[i])), prev_q2)
        
        c1, s1 = np.cos(q1[i]), np.sin(q1[i])
        c2, s2 = np.cos(q2[i]), np.sin(q2[i])
        B1 = x2.iloc[i]*c1*c2 + y2.iloc[i]*s1*c2 - z2.iloc[i]*s2
        B2 = -x2.iloc[i]*c1*s2 - y2.iloc[i]*s1*s2 - z2.iloc[i]*c2
        B3 = -x2.iloc[i]*s1 + y2.iloc[i]*c1
        
        q3[i] = resolve_sign(lambda: (np.arctan2(B3, B1), np.arctan2(-B3, B1)), prev_q3)
        
        # q4: forearm projection onto -u is (B2 + L1), not (B2 - L1)
        r_B = np.sqrt(B1**2 + B3**2)
        q4[i] = resolve_sign(lambda: (np.arctan2(r_B, B2 + L1), np.arctan2(-r_B, B2 + L1)), prev_q4)
        
        prev_q1, prev_q2, prev_q3, prev_q4 = q1[i], q2[i], q3[i], q4[i]
    
    return pd.DataFrame({
        'video_time_s': pose_df['video_time_s'],
        'q1': q1, 'q2': q2, 'q3': q3, 'q4': q4
    })

joint_angles = compute_joint_angles_from_data(pose_wide)
display(joint_angles.head())

,video_time_s,q1,q2,q3,q4
0,0.000000,1.218802,1.920924,-1.871925,1.634124
1,0.033333,1.219331,1.912324,-1.860677,1.635694
2,0.066667,1.219501,1.907185,-1.853209,1.637232
3,0.100000,1.219785,1.903611,-1.846121,1.639208
4,0.133333,1.221188,1.895549,-1.830999,1.642125


# Load and Process Data

In [4]:
# File paths (adjust if needed)
pose_file = f'camera0_{time_stamp}_poses_smoothed.csv'
emg_file = f'emg_data_{time_stamp}.csv'
emg_ts_file = f'emg_timestamps_{time_stamp}.csv'  # ts = timestamp

pose_raw = pd.read_csv(os.path.join(recordings_dir, pose_file))
emg_raw = pd.read_csv(os.path.join(recordings_dir, emg_file))
emg_ts = pd.read_csv(os.path.join(recordings_dir, emg_ts_file))

if DISPLAY_INTERMEDIATE:
    print("Pose shape:", pose_raw.shape)
    print("EMG shape:", emg_raw.shape)
    print("EMG timestamps shape:", emg_ts.shape)
    print("Pose head:")
    display(pose_raw[50:60])
    
    print("\nEMG head:")
    display(emg_raw.head(10))
    
    print("\nEMG timestamps head:")
    display(emg_ts.head(10))

Pose shape: (1683, 8)
EMG shape: (5600, 8)
EMG timestamps shape: (112, 2)
Pose head:


,frame_idx,video_time_s,position_name,source_tag_id,confidence,x,y,z
50,16,0.533333,shoulder,8,69.035667,-0.250150,-0.054898,1.555389
51,17,0.566667,bracelet,7,66.092667,0.130333,0.213166,1.496768
52,17,0.566667,elbow,1,72.475333,-0.134913,0.256143,1.427339
53,17,0.566667,shoulder,8,65.719000,-0.250192,-0.054694,1.555957
54,18,0.600000,bracelet,7,68.234333,0.131156,0.214752,1.497264
55,18,0.600000,elbow,1,74.005333,-0.134090,0.258276,1.432275
56,18,0.600000,shoulder,8,69.168333,-0.250009,-0.054317,1.555967
57,19,0.633333,bracelet,7,66.226667,0.132573,0.216840,1.498111
58,19,0.633333,elbow,1,71.452667,-0.132316,0.261999,1.441392
59,19,0.633333,shoulder,8,69.098667,-0.249537,-0.053795,1.555342



EMG head:


,Channel 1,Channel 2,Channel 3,Channel 4,Channel 5,Channel 6,Channel 7,Channel 8
0,0.0,0.0,409.305144,-430.919281,0.0,0.0,0.0,0.0
1,0.0,0.0,410.109807,-431.254558,0.0,0.0,0.0,0.0
2,0.0,0.0,408.411075,-430.695764,0.0,0.0,0.0,0.0
3,0.0,0.0,408.522833,-430.315784,0.0,0.0,0.0,0.0
4,0.0,0.0,411.540319,-430.807523,0.0,0.0,0.0,0.0
5,0.0,0.0,411.383857,-430.963985,0.0,0.0,0.0,0.0
6,0.0,0.0,409.774531,-430.941633,0.0,0.0,0.0,0.0
7,0.0,0.0,409.126331,-430.762819,0.0,0.0,0.0,0.0
8,0.0,0.0,410.310973,-431.254558,0.0,0.0,0.0,0.0
9,0.0,0.0,410.333325,-431.053392,0.0,0.0,0.0,0.0



EMG timestamps head:


,packet_index,pc_time
0,0,1.777099e+09
1,1,1.777099e+09
2,2,1.777099e+09
3,3,1.777099e+09
4,4,1.777099e+09
5,5,1.777099e+09
6,6,1.777099e+09
7,7,1.777099e+09
8,8,1.777099e+09
9,9,1.777099e+09


In [5]:
# We want one row per frame with all three body points.
# Pivot first to see missing pattern, then interpolate.

# 1. Pivot to wide format (same as before, but keep all frames)
pose_wide_raw = pose_raw.pivot_table(index=['frame_idx', 'video_time_s'],
                                     columns='position_name',
                                     values=['x', 'y', 'z'],
                                     aggfunc='mean')
pose_wide_raw.columns = [f'{pos}_{coord}' for coord, pos in pose_wide_raw.columns]
pose_wide_raw = pose_wide_raw.reset_index()

# 2. Count missing values per frame
missing_per_frame = pose_wide_raw.isna().sum(axis=1)
if DISPLAY_INTERMEDIATE:
    print(f"Frames with missing values:\n{missing_per_frame.value_counts().sort_index()}")

# 3. For each coordinate column, interpolate linearly over frame index
coord_cols = [c for c in pose_wide_raw.columns if c not in ['frame_idx', 'video_time_s']]
pose_wide = pose_wide_raw.copy()
pose_wide[coord_cols] = pose_wide[coord_cols].interpolate(method='linear', limit_direction='both', axis=0)

# 4. Verify no NaNs remain
remaining_nans = pose_wide.isna().sum().sum()
if DISPLAY_INTERMEDIATE:
    print(f"Remaining NaNs after interpolation: {remaining_nans}")
    display(pose_wide[10:20])

Frames with missing values:
0    550
3      6
Name: count, dtype: int64
Remaining NaNs after interpolation: 0


,frame_idx,video_time_s,bracelet_x,elbow_x,shoulder_x,bracelet_y,elbow_y,shoulder_y,bracelet_z,elbow_z,shoulder_z
10,10,0.333333,0.131093,-0.134188,-0.249718,0.211143,0.262703,-0.055073,1.500491,1.444166,1.554269
11,11,0.366667,0.131332,-0.133829,-0.249674,0.211557,0.258373,-0.054950,1.500884,1.432296,1.554194
12,12,0.400000,0.131350,-0.133819,-0.249710,0.211858,0.254664,-0.054898,1.500901,1.422983,1.554102
13,13,0.433333,0.131152,-0.133992,-0.249780,0.212058,0.253122,-0.054903,1.500463,1.419270,1.554047
14,14,0.466667,0.130785,-0.134248,-0.249877,0.212147,0.253694,-0.054939,1.499559,1.420673,1.554191
15,15,0.500000,0.130369,-0.134599,-0.250011,0.212169,0.254842,-0.054958,1.498325,1.423693,1.554672
16,16,0.533333,0.130129,-0.134937,-0.250150,0.212375,0.255459,-0.054898,1.497194,1.425558,1.555389
17,17,0.566667,0.130333,-0.134913,-0.250192,0.213166,0.256143,-0.054694,1.496768,1.427339,1.555957
18,18,0.600000,0.131156,-0.134090,-0.250009,0.214752,0.258276,-0.054317,1.497264,1.432275,1.555967
19,19,0.633333,0.132573,-0.132316,-0.249537,0.216840,0.261999,-0.053795,1.498111,1.441392,1.555342
